# \_\_missing\_\_ でキー依存デフォルト値を作成する方法を把握しておく

組み込み dict 型の setdefault メソッドは、状況によっては欠損キーの扱いをより短く書けます。
そのような状況の多くでは、組み込みモジュール collections の defaultdict 型の方がより良いツールとなります。
しかし、setdefault と defaultdict のどちらも正解でない場合があります。

例えば、ファイルシステムにあるソーシャルネットワークプロフィールの写真を管理するプログラムを書いているとします。
画像の読み書きができるようにプロフィールの写真のパス名をファイルハンドルにマップしてオープンするための辞書が必要です。
普通の dict インスタンスを使い、get メソッドと代入式を使ってキーがあるかチェックします。

In [ ]:
pictures = {}
path = '/path/to/profile-pic.png'

if (handle := pictures.get(path)) is None:
  try:
    handle = open(path, 'a+b')
  except OSError:
    print(f'Could not open file {path}')
    raise
  else:
    pictures[path] = handle

handle.seek(0)
image_data = handle.read()

辞書にファイルハンドルがある場合には、このコードは辞書に1回しかアクセスしません。
ファイルハンドルがない場合には、辞書は get で1回だけアクセスされ、try / except ブロック中の else 説で代入されます。
read メソッドの呼び出しは、open 呼び出しとその例外処理とははっきり区別されています。

これと同じロジックを in 式や keyError 例外の方式で使って実装することも可能ですが、その場合にはより多くの入れ子が深くなった辞書アクセスが必要です。他の選択肢でもよいことから、setdefault メソッドも動作すると期待するでしょう。

In [ ]:
try:
  handle = pictures.setdefault(path, open(path, 'a+b'))
except OSError:
  print(f'Could not open file {path}')
  raise
else:
  handle.seek(0)
  image_data = handle.read()

このコードには問題がたくさんあります。
ファイルハンドルを作る組み込み関数 open が、パスが辞書にある場合にも常に呼ばれます。これでは、その追加ファイルハンドルが同じプログラムにある既存のオープンハンドルと混乱する危険があります。open の呼び出しで例外が発生して、処理の必要があるかもしれませんが、それは同じ行の setdefault 呼び出しによる例外と区別できない可能性があります。

内部状態を管理するなら、このプロフィール写真の記録に defaultdict を使えるのではないかと考えるかもしれません。前と同じロジックを defaultdict クラスのヘルパー関数を使って実装してみました。

In [ ]:
from collections import defaultdict

def open_picture(profile_path):
  try:
    return open(profile_path, 'a+b')
  except OSError:
    print(f'Could not open file {profile_path}')
    raise

pictures = defaultdict(open_picture)
handle = pictures[path]
handle.seek(0)
image_data = handle.read()

問題は、defaultdict がコンストラクタに渡される関数には引数がないと仮定していることです。
これは、defaultdict が呼び出すヘルパー関数にはどのキーにあくせすしているかがわからないということを意味するので、open 呼び出しができません。この状況では、setdefault と defaultdict のどちらも役に立ちません。

幸い、このような状況は Python でよくあることなので、別の組み込みの解法が用意されています。
dict 型のサブクラスで特殊メソッド \_\_missing\_\_ を実装して欠損キーを扱うロジックを実装できます。これには、上で定義したのと同じ open_picture ヘルパーメソッドを使えます。

In [ ]:
class Pictures(dict):
  def __missing__(self, key):
    value = open_picture(key)
    self[key] = value
    return value

pictures = Pictures()
handle = pictures[path]
handle.seek(0)
image_data = handle.read()

pictures[path]辞書アクセスで、キーの path が辞書にないことがわかると、\_\_missing\_\_ メソッドが呼ばれます。これは、キーのデフォルト値を作り、辞書に挿入し、それを呼び出し元に返します。その後、同じ path でアクセスしたなら既に存在するので \_\_missing\_\_ は呼び出されません。

## 覚えておくこと

- dict の setdefault メソッドは、デフォルト値作成が高コストだったり例外が発生するような場合には使うべきではない。
- defaultdict に渡される関数には引数が渡せず、アクセスするキーに依存するデフォルト値が使えない
- メソッド \_\_missing\_\_ を持つ dict サブクラスを定義して、どのキーをアクセスしているかがわかるデフォルト値を作ることができる

## 補足

### \_\_missing\_\_ とは？

辞書に存在しないキーにアクセスしたときに自動的に呼び出される特別メソッドです。

In [2]:
class MyDict(dict):
    def __missing__(self, key):
        return 'not found'

上の例では、辞書にキーが無い場合、'not found'を返すようにできます。

### defaultdict との違い

標準では、キーが存在しない時のデフォルト値が必要な場合は、collections.defaultdictがよく使われます。

In [1]:
from collections import defaultdict

d = defaultdict(list)
d['a'].append(1)
print(d)  # {'a': [1]}

defaultdict(<class 'list'>, {'a': [1]})


しかし defaultdict は…

- 値がキーに依存していない（デフォルト値はいつも同じ型）
- d['b'] とアクセスした時点で 辞書内にキーが追加されてしまう

という特性があります。

In [6]:
# d['b'] とアクセスした時点で 辞書内にキーが追加されてしまう例

from collections import defaultdict

d = defaultdict(list)
print(d)  # {}
print(d['b'])  # []
print(d)  # {'b': []} ← 追加された！

defaultdict(<class 'list'>, {})
[]
defaultdict(<class 'list'>, {'b': []})


### \_\_missing\_\_ の利点

- キーに応じたデフォルト値が作れる
- 存在しないキー参照時に 辞書を汚染しない（キーが追加されない）

本書で紹介されている例：

> キーによって初期値が変わる場合（例：文字長でグループ分け）

In [3]:
class DefaultValueDict(dict):
    def __missing__(self, key):
        if isinstance(key, str):
            value = len(key)  # キーに依存したデフォルト値
            self[key] = value
            return value
        raise KeyError(key)

In [4]:
d = DefaultValueDict()
print(d['hello'])  # 5 （文字数）
print(d)           # {'hello': 5} ← 自動で追加された

5
{'hello': 5}


### よくある実用例

| 用途       | 例                       |
| -------- | ----------------------- |
| キーから動的生成 | ユーザーID → ユーザーの設定をDBから取得 |
| キーに基づく分類 | 文字長・接頭辞で初期値を決定          |
| 計算キャッシュ  | 計算結果を辞書に自動保存            |


In [5]:
class FibonacciCache(dict):
    def __missing__(self, key):
        if key < 2:
            return key
        value = self[key - 1] + self[key - 2]
        self[key] = value
        return value

fib = FibonacciCache()
print(fib[10])  # 55


55


### まとめ（覚えておくべきこと）

| 機能              | `defaultdict` | `__missing__` |
| --------------- | ------------- | ------------- |
| デフォルト値がキーに依存？   | ×             | ◎             |
| キー参照時に辞書へ追加される？ | ○             | 必須ではない        |
| 実装の簡単さ          | ◎             | △（クラスが必要）     |

- キー依存ロジックが必要な場合は、\_\_missing\_\_ が最適
- シンプルな「キー無い時は空リスト」のような用途には defaultdict
